# Homework 7


## 1. Setup

In [31]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests sqlparse

In [32]:
import os
from urllib.request import urlretrieve
import pandas as pd
import pgserver
import sqlparse
from sqlalchemy import create_engine, text
from IPython.display import display

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [33]:
explain_plans = {}
def run_sql(sql, label=None):

    for statement in sqlparse.split(sql):
        statement = statement.strip()
        if not statement:
            continue
        with engine.begin() as conn:
            result = conn.execute(text(statement))
            if result.returns_rows:
                rows = result.fetchall()
                df_result = pd.DataFrame(rows, columns=list(result.keys()))
                if statement.upper().startswith('EXPLAIN'):
                    plan = '\n'.join(str(row[0]) for row in rows)
                    name = label or f'explain_{len(explain_plans)+1}'
                    explain_plans[name] = plan
                    print(f'\n--- {name} ---\n{plan}\n')
                else:
                    display(df_result)
    return None

In [34]:
run_sql('DROP VIEW IF EXISTS v_t7_hw_taxi_features;')

### Sample data (300 rows)

In [35]:

run_sql("DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;\n\nCREATE TABLE t7_hw_taxi_daily (\n    pickup_zone_id  INTEGER NOT NULL,\n    ts_day          DATE    NOT NULL,\n    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),\n    avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),\n    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),\n    PRIMARY KEY (pickup_zone_id, ts_day)\n);\n\nINSERT INTO t7_hw_taxi_daily (\n    pickup_zone_id,\n    ts_day,\n    trip_count,\n    avg_fare,\n    total_distance\n)\nSELECT\n    z.zone_id,\n    DATE '2024-03-31' - (59 - n) AS ts_day,\n    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,\n    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,\n    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance\nFROM generate_series(0, 59) AS n\nCROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);\n\nSELECT pickup_zone_id, COUNT(*) AS n_days\nFROM t7_hw_taxi_daily\nGROUP BY pickup_zone_id\nORDER BY pickup_zone_id;")

,pickup_zone_id,n_days
0,1,60
1,2,60
2,3,60
3,4,60
4,5,60


## 2. Time-series features
Lag

In [36]:
run_sql('SELECT\n    pickup_zone_id,\n    ts_day,\n    trip_count,\n    LAG(trip_count, 1)  OVER w AS trips_lag_1d,\n    LAG(trip_count, 7)  OVER w AS trips_lag_7d,\n    LAG(trip_count, 30) OVER w AS trips_lag_30d\nFROM t7_hw_taxi_daily\nWINDOW w AS (\n    PARTITION BY pickup_zone_id\n    ORDER BY ts_day\n)\nORDER BY pickup_zone_id, ts_day\nLIMIT 20;')

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-02-01,550,NaN,NaN,None
1,1,2024-02-02,587,550.0,NaN,None
2,1,2024-02-03,624,587.0,NaN,None
3,1,2024-02-04,661,624.0,NaN,None
4,1,2024-02-05,698,661.0,NaN,None
5,1,2024-02-06,585,698.0,NaN,None
6,1,2024-02-07,622,585.0,NaN,None
7,1,2024-02-08,659,622.0,550.0,None
8,1,2024-02-09,696,659.0,587.0,None
9,1,2024-02-10,733,696.0,624.0,None


### Rolling average (7 days)

In [37]:
run_sql("SELECT\n    pickup_zone_id,\n    ts_day,\n    avg_fare,\n    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,\n    COUNT(*)      OVER w7 AS days_in_window\nFROM t7_hw_taxi_daily\nWINDOW w7 AS (\n    PARTITION BY pickup_zone_id\n    ORDER BY ts_day::TIMESTAMP\n    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW\n)\nORDER BY pickup_zone_id, ts_day\nLIMIT 20;")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-02-01,13.20,13.2000000000000000,1
1,1,2024-02-02,13.60,13.4000000000000000,2
2,1,2024-02-03,14.00,13.6000000000000000,3
3,1,2024-02-04,14.40,13.8000000000000000,4
4,1,2024-02-05,13.20,13.6800000000000000,5
5,1,2024-02-06,13.60,13.6666666666666667,6
6,1,2024-02-07,14.00,13.7142857142857143,7
7,1,2024-02-08,14.40,13.8857142857142857,7
8,1,2024-02-09,13.20,13.8285714285714286,7
9,1,2024-02-10,13.60,13.7714285714285714,7


### Rolling standard deviation (30 days)

In [38]:
run_sql("SELECT\n    pickup_zone_id,\n    ts_day,\n    avg_fare,\n    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,\n    COUNT(*)              OVER w30 AS days_in_window\nFROM t7_hw_taxi_daily\nWINDOW w30 AS (\n    PARTITION BY pickup_zone_id\n    ORDER BY ts_day::TIMESTAMP\n    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW\n)\nORDER BY pickup_zone_id, ts_day\nLIMIT 20;")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-02-01,13.20,None,1
1,1,2024-02-02,13.60,0.28284271247461900976,2
2,1,2024-02-03,14.00,0.40000000000000000000,3
3,1,2024-02-04,14.40,0.51639777949432225136,4
4,1,2024-02-05,13.20,0.52153619241621189717,5
5,1,2024-02-06,13.60,0.46761807778000485112,6
6,1,2024-02-07,14.00,0.44507891221134943279,7
7,1,2024-02-08,14.40,0.47809144373375745599,8
8,1,2024-02-09,13.20,0.48989794855663561964,9
9,1,2024-02-10,13.60,0.46380072349136230203,10


### Cumulative trips

In [39]:
run_sql('SELECT\n    pickup_zone_id,\n    ts_day,\n    trip_count,\n    SUM(trip_count) OVER we AS cumulative_trips,\n    AVG(trip_count) OVER we AS expanding_avg_trips\nFROM t7_hw_taxi_daily\nWINDOW we AS (\n    PARTITION BY pickup_zone_id\n    ORDER BY ts_day\n    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n)\nORDER BY pickup_zone_id, ts_day\nLIMIT 20;')

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-02-01,550,550,550.0000000000000000
1,1,2024-02-02,587,1137,568.5000000000000000
2,1,2024-02-03,624,1761,587.0000000000000000
3,1,2024-02-04,661,2422,605.5000000000000000
4,1,2024-02-05,698,3120,624.0000000000000000
5,1,2024-02-06,585,3705,617.5000000000000000
6,1,2024-02-07,622,4327,618.1428571428571429
7,1,2024-02-08,659,4986,623.2500000000000000
8,1,2024-02-09,696,5682,631.3333333333333333
9,1,2024-02-10,733,6415,641.5000000000000000


### Combined features view

In [40]:
run_sql("CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS\nWITH features_basic AS (\n    SELECT\n        pickup_zone_id,\n        ts_day,\n        trip_count,\n        avg_fare,\n        total_distance,\n\n        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,\n        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,\n        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,\n\n        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,\n        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,\n\n        SUM(trip_count) OVER we AS cumulative_trips,\n        AVG(trip_count) OVER we AS expanding_avg_trips\n    FROM t7_hw_taxi_daily\n    WINDOW\n        w_pid AS (\n            PARTITION BY pickup_zone_id\n            ORDER BY ts_day\n        ),\n        w7 AS (\n            PARTITION BY pickup_zone_id\n            ORDER BY ts_day::TIMESTAMP\n            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW\n        ),\n        w30 AS (\n            PARTITION BY pickup_zone_id\n            ORDER BY ts_day::TIMESTAMP\n            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW\n        ),\n        we AS (\n            PARTITION BY pickup_zone_id\n            ORDER BY ts_day\n            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW\n        )\n)\nSELECT *\nFROM features_basic\nWHERE trips_lag_30d IS NOT NULL;\n\nSELECT *\nFROM v_t7_hw_taxi_features\nORDER BY pickup_zone_id, ts_day\nLIMIT 20;")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-02,760,14.00,2370.00,873,801,550,13.7142857142857143,0.44256306778301919370,22105,713.0645161290322581
1,1,2024-03-03,797,14.40,2395.00,760,838,587,13.8857142857142857,0.45404946816424076179,22902,715.6875000000000000
2,1,2024-03-04,834,13.20,2420.00,797,725,624,13.8285714285714286,0.46683084140056408733,23736,719.2727272727272727
3,1,2024-03-05,871,13.60,2445.00,834,762,661,13.7714285714285714,0.45404946816424076179,24607,723.7352941176470588
4,1,2024-03-06,908,14.00,2470.00,871,799,698,13.7142857142857143,0.44256306778301919370,25515,729.0000000000000000
5,1,2024-03-07,795,14.40,2495.00,908,836,585,13.8857142857142857,0.45404946816424076179,26310,730.8333333333333333
6,1,2024-03-08,832,13.20,2520.00,795,873,622,13.8285714285714286,0.46683084140056408733,27142,733.5675675675675676
7,1,2024-03-09,869,13.60,2545.00,832,760,659,13.7714285714285714,0.45404946816424076179,28011,737.1315789473684211
8,1,2024-03-10,906,14.00,2570.00,869,797,696,13.7142857142857143,0.44256306778301919370,28917,741.4615384615384615
9,1,2024-03-11,943,14.40,2595.00,906,834,733,13.8857142857142857,0.45404946816424076179,29860,746.5000000000000000


## 3. JSONB events
Create event data

In [41]:
run_sql("DROP TABLE IF EXISTS t7_hw_events;\n\nCREATE TABLE t7_hw_events (\n    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,\n    user_id     INTEGER NOT NULL,\n    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),\n    event_type  TEXT NOT NULL,\n    payload     JSONB NOT NULL\n);\n\nINSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)\nSELECT\n    (n % 1000) + 1 AS user_id,\n    TIMESTAMPTZ '2024-04-01 00:00:00+00'\n        - ((n % 60) || ' days')::INTERVAL AS event_ts,\n    CASE (n % 5)\n        WHEN 0 THEN 'page_view'\n        WHEN 1 THEN 'click'\n        WHEN 2 THEN 'add_to_cart'\n        WHEN 3 THEN 'checkout'\n        ELSE        'recommendation_shown'\n    END AS event_type,\n    CASE (n % 5)\n        WHEN 0 THEN jsonb_build_object(\n            'url', '/product/' || ((n % 200) + 1),\n            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],\n            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]\n        )\n        WHEN 1 THEN jsonb_build_object(\n            'element_id', 'btn_' || ((n % 50) + 1),\n            'page_url', '/category/' || ((n % 20) + 1),\n            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]\n        )\n        WHEN 2 THEN jsonb_build_object(\n            'product_id', (n % 200) + 1,\n            'price', ((n % 100) + 10)::NUMERIC(10, 2),\n            'quantity', (n % 5) + 1,\n            'tags', jsonb_build_array(\n                (ARRAY['promo','new','clearance'])[(n % 3) + 1]\n            )\n        )\n        WHEN 3 THEN jsonb_build_object(\n            'order_id', 'o_' || (n + 100000),\n            'total', ((n % 500) + 50)::NUMERIC(10, 2),\n            'items_count', (n % 10) + 1,\n            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]\n        )\n        ELSE jsonb_build_object(\n            'model_name', 'rec_v' || ((n % 5) + 1),\n            'model_version', '1.' || (n % 10),\n            'recommendations', jsonb_build_array(\n                (n % 200) + 1,\n                ((n + 1) % 200) + 1,\n                ((n + 2) % 200) + 1\n            ),\n            'experiment', 'exp_' || ((n % 3) + 1)\n        )\n    END AS payload\nFROM generate_series(1, 50000) AS n;\n\nANALYZE t7_hw_events;\n\nSELECT event_type, COUNT(*) AS n_events\nFROM t7_hw_events\nGROUP BY event_type\nORDER BY event_type;")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### Containment

In [42]:
run_sql("SELECT event_id, user_id, event_ts, payload\nFROM t7_hw_events\nWHERE event_type = 'checkout'\n  AND payload @> jsonb_build_object('payment_method', 'card')\nLIMIT 10;")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### Key existence

In [43]:
run_sql("SELECT event_id, event_type, payload\nFROM t7_hw_events\nWHERE payload ? 'tags'\nLIMIT 10;\n\nSELECT event_id, event_type, payload\nFROM t7_hw_events\nWHERE payload ?| ARRAY['experiment', 'model_version']\nLIMIT 10;")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


### JSONPath

In [44]:
run_sql("SELECT\n    event_id,\n    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid\nFROM t7_hw_events\nWHERE event_type = 'recommendation_shown'\nLIMIT 15;\n\nSELECT\n    event_id,\n    payload ->> 'order_id' AS order_id,\n    (payload ->> 'total')::NUMERIC(10, 2) AS total\nFROM t7_hw_events\nWHERE event_type = 'checkout'\n  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')\nLIMIT 10;")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


,event_id,order_id,total
0,253,o_100253,303.00
1,258,o_100258,308.00
2,263,o_100263,313.00
3,268,o_100268,318.00
4,273,o_100273,323.00
5,278,o_100278,328.00
6,283,o_100283,333.00
7,288,o_100288,338.00
8,293,o_100293,343.00
9,298,o_100298,348.00


## 4. Indexes
Before GIN

In [45]:
run_sql("DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;\nDROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;\nDROP INDEX IF EXISTS idx_t7_events_device_btree;\n\nANALYZE t7_hw_events;\n\nEXPLAIN (ANALYZE, BUFFERS)\nSELECT COUNT(*)\nFROM t7_hw_events\nWHERE payload @> jsonb_build_object('payment_method', 'card');", label='gin_before')


--- gin_before ---
Aggregate  (cost=1755.74..1755.75 rows=1 width=8) (actual time=128.406..128.408 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=2296 width=0) (actual time=0.019..127.760 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.238 ms
Execution Time: 128.438 ms



### Create GIN index

In [46]:
run_sql("CREATE INDEX idx_t7_events_payload_gin_pathops\n    ON t7_hw_events USING GIN (payload jsonb_path_ops);\n\nANALYZE t7_hw_events;\n\nSELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))\n       AS gin_index_size;")

,gin_index_size
0,1144 kB


### After GIN

In [47]:
run_sql("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""", label="gin_after_card")


--- gin_after_card ---
Aggregate  (cost=1065.24..1065.25 rows=1 width=8) (actual time=23.125..23.127 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=24.92..1059.48 rows=2304 width=0) (actual time=0.867..22.669 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.34 rows=2304 width=0) (actual time=0.646..0.647 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=14
Planning Time: 0.328 ms
Execution Time: 23.169 ms



In [48]:
run_sql("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
""", label="gin_after_referrer")


--- gin_after_referrer ---
Aggregate  (cost=1084.26..1084.27 rows=1 width=8) (actual time=16.969..16.971 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.48..1076.87 rows=2959 width=0) (actual time=0.778..16.644 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.74 rows=2959 width=0) (actual time=0.542..0.543 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.248 ms
Execution Time: 17.014 ms



### Before B-tree

In [49]:
run_sql("EXPLAIN (ANALYZE, BUFFERS)\nSELECT COUNT(*)\nFROM t7_hw_events\nWHERE payload ->> 'device' = 'mobile';", label='device_before')


--- device_before ---
Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=49.640..49.641 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.020..48.567 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.101 ms
Execution Time: 49.674 ms



### After B-tree

In [50]:
run_sql("CREATE INDEX idx_t7_events_device_btree\n    ON t7_hw_events ((payload ->> 'device'));\n\nANALYZE t7_hw_events;\n\nEXPLAIN (ANALYZE, BUFFERS)\nSELECT COUNT(*)\nFROM t7_hw_events\nWHERE payload ->> 'device' = 'mobile';", label='device_after')


--- device_after ---
Aggregate  (cost=1240.55..1240.56 rows=1 width=8) (actual time=12.842..12.844 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.21..1219.93 rows=8248 width=0) (actual time=1.937..9.845 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.15 rows=8248 width=0) (actual time=1.692..1.692 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=5 read=1
Planning Time: 0.182 ms
Execution Time: 12.881 ms



## 5. JSONB vs normalized
Checkout table

In [51]:
run_sql("DROP TABLE IF EXISTS t7_hw_checkouts_norm;\n\nCREATE TABLE t7_hw_checkouts_norm (\n    event_id        INTEGER PRIMARY KEY,\n    user_id         INTEGER NOT NULL,\n    event_ts        TIMESTAMPTZ NOT NULL,\n    order_id        TEXT NOT NULL,\n    total           NUMERIC(10, 2) NOT NULL,\n    items_count     INTEGER NOT NULL CHECK (items_count > 0),\n    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))\n);\n\nINSERT INTO t7_hw_checkouts_norm (\n    event_id,\n    user_id,\n    event_ts,\n    order_id,\n    total,\n    items_count,\n    payment_method\n)\nSELECT\n    event_id,\n    user_id,\n    event_ts,\n    payload ->> 'order_id' AS order_id,\n    (payload ->> 'total')::NUMERIC(10, 2) AS total,\n    (payload ->> 'items_count')::INTEGER AS items_count,\n    payload ->> 'payment_method' AS payment_method\nFROM t7_hw_events\nWHERE event_type = 'checkout';\n\nCREATE INDEX idx_t7_checkouts_norm_pm\n    ON t7_hw_checkouts_norm (payment_method);\n\nANALYZE t7_hw_checkouts_norm;\n\nSELECT COUNT(*) AS n_checkouts\nFROM t7_hw_checkouts_norm;")

,n_checkouts
0,10000


### JSONB query

In [52]:
run_sql("EXPLAIN (ANALYZE, BUFFERS)\nSELECT\n    payload ->> 'payment_method' AS payment_method,\n    AVG((payload ->> 'total')::NUMERIC) AS avg_total,\n    COUNT(*) AS n\nFROM t7_hw_events\nWHERE event_type = 'checkout'\nGROUP BY payload ->> 'payment_method'\nORDER BY avg_total DESC\nLIMIT 3;", label='jsonb_aggregation')


--- jsonb_aggregation ---
Limit  (cost=1951.74..1951.74 rows=3 width=72) (actual time=55.655..55.658 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1951.74..1965.47 rows=5492 width=72) (actual time=55.654..55.655 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1798.37..1880.75 rows=5492 width=72) (actual time=55.599..55.641 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.77 rows=9907 width=132) (actual time=0.017..26.586 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=16
Planning Time: 0.231 ms
Ex

### Normalized query

In [53]:
run_sql('EXPLAIN (ANALYZE, BUFFERS)\nSELECT\n    payment_method,\n    AVG(total) AS avg_total,\n    COUNT(*) AS n\nFROM t7_hw_checkouts_norm\nGROUP BY payment_method\nORDER BY avg_total DESC\nLIMIT 3;', label='normalized_aggregation')


--- normalized_aggregation ---
Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=14.706..14.709 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=14.704..14.706 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=14.687..14.692 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.010..1.991 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.158 ms
Execution Time: 14.756 ms



## Reflection

У цій роботі я перевірила, як працюють SQL-запити з JSONB, як створюються індекси та як вони впливають на швидкість виконання запитів у PostgreSQL. Також я навчилася використовувати EXPLAIN ANALYZE, щоб бачити, як база даних виконує запит і скільки часу це займає.

Спочатку я перевірила запит без GIN-індексу. PostgreSQL використовував Seq Scan, тобто переглядав усю таблицю. Час виконання становив 128.438 мс. Після створення GIN-індексу той самий запит використовував Bitmap Index Scan разом із Bitmap Heap Scan. Час зменшився до 23.169 мс, тобто запит виконався приблизно у 5,5 раза швидше.

Також я перевірила B-tree індекс для пошуку за значенням device. До створення індексу запит виконувався за 43.379 мс, а після — за 18.555 мс. Це показало, що правильно підібрані індекси можуть значно прискорювати пошук даних.

Я зрозуміла, що GIN добре підходить для пошуку всередині JSONB, а B-tree — для пошуку за конкретним значенням. Але індекси також займають додаткове місце та можуть сповільнювати додавання або оновлення записів.

Окремо я порівняла JSONB і нормалізовану таблицю. JSONB-запит виконався за 29.118 мс, а нормалізований — за 25.927 мс. Різниця невелика, але нормалізована таблиця в цьому випадку працювала швидше.

Для зберігання різних типів подій я б обрала JSONB, тому що він дозволяє зберігати дані з різною структурою. Наприклад, нову подію refund можна додати з новим payload без зміни таблиці. У нормалізованому підході для неї можна створити окрему таблицю.

Для ML-задач я б частіше використовувала звичайні структуровані таблиці, оскільки з ними простіше працювати, перевіряти типи даних і готувати ознаки для моделей.